## WCUS Trips cleaning steps
Get the data [here](https://ndclibrary.sec.usace.army.mil/resource?title=2023%20WCUS%20Trips%20-%20All%20Regions%20&documentId=07ddf14b-1522-4c6f-d894-40cd74d58a7f).

1. Filter to get only In/Out/Thru=="Outbound Shipping" or "Inbound Shipping"
2. Filter to get only TrafficCode==1 (aka TrafficName=="Domestic (Trips & Drafts)")
3. Keep only the columns:
    * 'RegionName', 'Up/Down', 'VesselType', 'VesselTypeName', 'VesselDraftFt', 'Trips', 'CompletedYear'
4. Create a few separate data sets: 
    * Filter to get only WaterwayCode==3924 (aka WaterwayName=="Duluth-Superior, MN and WI")
    * Filter to get only RegionName=="GREAT LAKES"
5. Save as something descriptive like "WCUS_Trips_DuluthSuperior_Outbound_Domestic_2014-2023.csv"

## List of Top Iron Ore Outports and Inports (USACE focus)
There are more ports that see iron ore than the ones selected here, but I have chosen to just focus on Corps ports due to better documentation and to scope the project.

Corps ports were selected based on [this digital brochure](https://lre-ops.usace.army.mil/OandM/GLNAV/Main_Page/NavSystemBrochure.pdf).

In [ ]:
import pandas as pd
# Taconite and Silver Bay (last two in inline comment) are not USACE ports

# Careful! Cities and their harbors are listed separately.
# E.g. 3841 for Marquette Harbor (admin, maintenance) vs. 3844 for Marquette Township (other)
# 3619 is the Presque Isle farther east, by Alpena
# 3845 Presque Isle, MI is just a subset of Marquette, so exclude it.
outports = pd.DataFrame({
    'WaterwayCode':[3924, 3926, 3841]#, 3845, 3929, 3928]
    })

# Make sure to distinguish between receipts and throughports
    # E.g. Chicago is just a rail hub, other places go straight to mills nearby
inports = pd.DataFrame({
    'WaterwayCode': [3738, 3736, 3739, 3204, 3217, 3315, 3220, 3219, 3741, 3747]
})

## WCUS Trips Cleaning

In [ ]:
path = "../data/raw/Trips_AllRegions_10yr_2014-2023.xlsx"

wcus = pd.read_excel(path, sheet_name="Trips_AllRegions_10yr_2014-2023")

In [ ]:
# Check we have selected the right ports
wcus.merge(outports, how='right').loc[:, ['WaterwayName', 'WaterwayCode']].drop_duplicates()

In [ ]:
min_draft_ft = 25

wcus_all_out = wcus.loc[
        wcus['WaterwayCode'].isin(outports['WaterwayCode']) 
            & wcus['VesselTypeName'].isin(['Self-Propelled dry'])
            & (wcus['In/Out/Thru'] == 'Outbound Shipping')
            & (wcus['VesselDraftFt'] >= min_draft_ft)
            & (wcus['TrafficCode'] == 1) # Domestic (Trips & Drafts)
            ,
        :
    ]
# wcus_all_out.head(3)

In [ ]:
cols_keep = ['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'VesselTypeName', 'VesselDraftFt', 'Trips', 'CompletedYear']



### Outbound Trip Counts (25ft+ draft)

In [ ]:
# outports_dict = dict(zip(outports['WaterwayCode'], ['duluth_superior', 'two_harbors', 'presque_isle', 'marquette_harbor']))
outports_dict = dict(zip(outports['WaterwayCode'], ['duluth_superior', 'two_harbors', 'marquette_harbor']))


In [ ]:
wcus_outport_counts = wcus.loc[
    wcus['WaterwayCode'].isin(outports['WaterwayCode']) 
            & wcus['VesselTypeName'].isin(['Self-Propelled dry'])
            & (wcus['In/Out/Thru'] != 'Waterway')
            & (wcus['VesselDraftFt'] >= min_draft_ft)
            & (wcus['Up/Down']=='Port')
            & (wcus['TrafficCode'] == 1),
    ['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'CompletedYear', 'Trips']
].groupby(['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'CompletedYear']).sum().reset_index()

wcus_outport_counts.to_csv(f'../data/clean/trip_counts/wcus_outport_{min_draft_ft}ftplus_counts.csv', index=False)

### Inbound Trip Counts (25ft+ draft)

In [ ]:
wcus_inport_counts = wcus.loc[
    wcus['WaterwayCode'].isin(inports['WaterwayCode']) 
            & wcus['VesselTypeName'].isin(['Self-Propelled dry'])
            & (wcus['In/Out/Thru'] != 'Waterway')
            & (wcus['VesselDraftFt'] >= min_draft_ft)
            & (wcus['Up/Down']=='Port')
            & (wcus['TrafficCode'] == 1),
    ['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'CompletedYear', 'Trips']
].groupby(['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'CompletedYear']).sum().reset_index()

In [ ]:
wcus_inport_counts.to_csv(f'../data/clean/trip_counts/wcus_inport_{min_draft_ft}ftplus_counts.csv', index=False)

### Soo Locks through-trips

In [ ]:
# This is the Sault Ste. Marie river reach, which is not intended for tracking thru-traffic as a waterway
wcus.loc[wcus['WaterwayName'].str.startswith('Sault') , :].head(3)


In [ ]:
wcus_soo = wcus.loc[
        wcus['WaterwayCode'].isin([3811]) 
            & wcus['VesselTypeName'].isin(['Self-Propelled dry'])
            & (wcus['In/Out/Thru'] == 'Waterway')
            & (wcus['VesselDraftFt'] >= min_draft_ft)
            & (wcus['TrafficCode'] == 1), 
        :
    ]

wcus_soo_counts = wcus_soo.loc[
    :,
    ['WaterwayCode', 'WaterwayName', 'Up/Down', 'CompletedYear', 'Trips']
].groupby(['WaterwayCode', 'WaterwayName', 'Up/Down', 'CompletedYear']).sum().reset_index()

In [ ]:
wcus_soo_counts.to_csv(f'../data/clean/trip_counts/wcus_soo_{min_draft_ft}ftplus_counts.csv', index=False)

## WCUS Cargo cleaning steps

1. Filter to only WaterwayCodes in `inports` and `outports`
2. `CommodityCode` 4410 (iron ore)
3. Only Lakewise vessel movements (no intraport movements, for example)

## WCUS Cargo cleaning

In [ ]:
path = "../data/raw/Cargo_AllRegions_10yr_2013-2022.xlsx"

cargo = pd.read_excel(path, sheet_name="Cargo_AllRegions_10yr_2013-2022")

In [ ]:
cols_to_drop_cargo = ['TrafficCode', 'Allo1Code', 'Allo2Code', 'Up/Down', 'TonMiles']

### Outbound

In [ ]:
cargoes_outport = cargo.loc[
            (cargo['WaterwayCode'].isin(outports['WaterwayCode']))
            & (cargo['CommodityCode']==4410)
            & (cargo['TrafficName']=='Lakewise')
            & (cargo['In/Out/Thru']=='Outbound Shipping'), 
            :
        ].drop(columns = cols_to_drop_cargo)
# cargoes_outport.head()

In [ ]:
cargoes_outport.to_csv('../data/clean/cargoes/wcus_outport_cargoes.csv')

In [ ]:
# cargo.loc[(cargo['WaterwayCode']==3619) & (cargo['CommodityName']=='Iron Ore'), :]

### Inbound

In [ ]:
# Which inport has historically received the most ore on average?
cargo.loc[
    cargo['WaterwayCode'].isin(inports['WaterwayCode'])
        & (cargo['CommodityCode']==4410)
        & (cargo['TrafficName']=='Lakewise')
        & (cargo['In/Out/Thru']=='Inbound Receiving')
    ,
    ['WaterwayCode', 'WaterwayName', 'ShortTons']
].groupby(by=['WaterwayCode', 'WaterwayName']).mean().sort_values(by='ShortTons', ascending=False)

In [ ]:
cargoes_inport = cargo.loc[
            (cargo['WaterwayCode'].isin(inports['WaterwayCode']))
            & (cargo['CommodityCode']==4410)
            & (cargo['TrafficName']=='Lakewise')
            & (cargo['In/Out/Thru']=='Inbound Receiving'), 
            :
        ].drop(columns = cols_to_drop_cargo)
# cargoes_inport.head()

In [ ]:
cargoes_inport.to_csv('../data/clean/cargoes/wcus_inport_cargoes.csv')

### Soo Locks

In [ ]:
cargoes_soo = cargo.loc[
            cargo['WaterwayCode'].isin([3811])
                & (cargo['CommodityCode']==4410)
                # & (cargo['In/Out/Thru']=='Inbound Receiving')
                & (cargo['TrafficName']=='Lakewise'),
            :
        ]#.drop(columns = cols_to_drop_cargo)
# cargoes_soo.head()

In [ ]:
cargoes_soo.to_csv('../data/clean/cargoes/wcus_soo_cargoes.csv')